In [0]:
# ============================================================
# Notebook: 01_bronze_online_retail_ingestion
# Purpose: Load Online Retail II CSV from Unity Catalog Volume
#          into Bronze Delta table while preserving source schema
# Author: Dele Fatoba
# ============================================================

from pyspark.sql.functions import current_timestamp, col

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retaildataops_dbw_dev_v4ptce_7405619702729069.raw;

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:439)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:439)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
%sql
USE CATALOG retaildataops_dbw_dev_v4ptce_7405619702729069;
USE SCHEMA raw;

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS online_retail_volume
LOCATION 'abfss://raw@retaildataopsdevv4ptce.dfs.core.windows.net/';

In [0]:
%sql
SHOW VOLUMES IN retaildataops_dbw_dev_v4ptce_7405619702729069.raw;

database,volume_name
raw,online_retail_volume


In [0]:
# Source file in Unity Catalog Volume
# This volume points to ADLS Gen2 through:
# Storage Credential -> External Location -> External Volume -> ADLS Gen2(raw container)

catalog_name = "retaildataops_dbw_dev_v4ptce_7405619702729069"
volume_name = "online_retail_volume"

catalog_name = "retaildataops_dbw_dev_v4ptce_7405619702729069"
volume_name = "online_retail_volume"

source_volume_path = (
    f"/Volumes/{catalog_name}/raw/{volume_name}/"
)

source_file_path = (
    f"{source_volume_path}online_retail_II.csv"
)

bronze_table_name = (
    f"{catalog_name}.bronze.online_retail_transactions"
)

# Validate source file path

display(
    dbutils.fs.ls(source_volume_path)
)

path,name,size,modificationTime
dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,online_retail_II.csv,94850204,1780400967000


In [0]:
# Read CSV from Unity Catalog Volume

bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .csv(source_file_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)
display(bronze_df.limit(20))
bronze_df.printSchema()

print(f"Bronze record count: {bronze_df.count():,}")




Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,ingestion_timestamp,source_file
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom,2026-06-02T14:16:49.638Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv


root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

Bronze record count: 1,067,371


In [0]:
# Create Bronze schema if it does not exist

spark.sql("""
CREATE SCHEMA IF NOT EXISTS
retaildataops_dbw_dev_v4ptce_7405619702729069.bronze
""")

DataFrame[]

In [0]:
# Write Bronze Delta table as a Unity Catalog managed table

spark.sql(f"DROP TABLE IF EXISTS {bronze_table_name}")

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .saveAsTable(bronze_table_name)
)

print(f"Bronze Delta table created with column mapping enabled: {bronze_table_name}")




Bronze Delta table created with column mapping enabled: retaildataops_dbw_dev_v4ptce_7405619702729069.bronze.online_retail_transactions


In [0]:
# Verify Bronze table

display(
    spark.table(bronze_table_name)
    .limit(20)
)

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,ingestion_timestamp,source_file
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
